# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing the metadata object directly
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their @id

print("Available Record Sets:")
record_sets = metadata.recordSet
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For this dataset, find the main survey record set
survey_rs_id = None
for rs in record_sets:
    if 'survey' in rs.get('name','').lower() or 'mental health' in rs.get('name','').lower():
        survey_rs_id = rs['@id']

if not survey_rs_id:
    survey_rs_id = record_sets[0]['@id']
print(f"\nSelected RecordSet @id for analysis: {survey_rs_id}\n")

# List fields and columns in the selected record set
fields = [fld['@id'] for fld in rs.get('field', [])]
print("Fields in selected RecordSet:")
for fld in rs.get('field', []):
    print(f"- Field @id: {fld['@id']}, name: {fld.get('name', 'N/A')}, dataType: {fld.get('dataType', 'N/A')}")

# If the record set contains columns
if 'column' in rs:
    print("\nColumns in selected RecordSet:")
    for col in rs.get('column', []):
        print(f"- Column @id: {col['@id']}, name: {col.get('name', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from specified record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show columns from selected main survey record set
print(f"Available columns in DataFrame for RecordSet {survey_rs_id}:")
if survey_rs_id in dataframes:
    print(dataframes[survey_rs_id].columns.tolist())
    dataframes[survey_rs_id].head()
else:
    print(f"RecordSet {survey_rs_id} data not loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis
# Use the GAD-7 score, which measures anxiety symptoms, common in mental health datasets
# Reference the actual @id from the schema (example: 'https://api.app.sen.science/frontiers/7160411/gad7_score')

numeric_field_id = None
fields_to_search = ['gad7', 'phq9', 'psq', 'score']

for col in dataframes[survey_rs_id].columns:
    for search in fields_to_search:
        if search in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break
        
if not numeric_field_id:
    # Default to the first numeric-looking column
    for col in dataframes[survey_rs_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[survey_rs_id][col]):
            numeric_field_id = col
            break

print(f"Numeric field selected: {numeric_field_id}")

# Use threshold to filter high symptom scores (e.g., GAD-7 >= 10 indicates moderate/severe anxiety)
threshold = 10
filtered_df = dataframes[survey_rs_id][dataframes[survey_rs_id][numeric_field_id] >= threshold]
print(f"Filtered records with {numeric_field_id} >= {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a demographic field (e.g., 'level_of_education' or 'marital_status')
# Reference by @id (column name in DataFrame)
group_field_candidates = ['level_of_education', 'marital_status', 'gender']
group_field = None
for col in dataframes[survey_rs_id].columns:
    for search in group_field_candidates:
        if search in col.lower():
            group_field = col
            break
    if group_field:
        break

print(f"Grouping by field: {group_field}")

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization using matplotlib and seaborn
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(dataframes[survey_rs_id][numeric_field_id], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Compare mean scores across grouped field
if group_field:
    plt.figure(figsize=(8, 5))
    sns.barplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"Average {numeric_field_id} by {group_field} (filtered)")
    plt.xticks(rotation=30)
    plt.ylabel(f"Average {numeric_field_id}")
    plt.xlabel(f"{group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load a Croissant-based dataset with `mlcroissant`, referencing all entities via their `@id` fields.
- Key survey record sets and fields were identified using their `@id`.
- Data was extracted and analyzed for high symptom scores, and normalized distributions were visualized.
- Demographic attributes were used for grouping, revealing patterns in mental health indicators.
- The FAIR² Mental Health Survey Dataset offers a structured foundation for research into mental health and demographic trends in Kilifi, Kenya.